# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an end-to-end example for loading and exploring the FAIR^2 clinical colorectal cancer dataset using the `mlcroissant` library. We will use only Croissant `@id` fields to reference entities such as record sets and fields for consistency.

### Dataset Source
The dataset source is defined by its Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading

Load the dataset Croissant metadata and explore its contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object, not a dict)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")

## 2. Data Overview

Let’s list available record sets and their fields, with `@id`s for reference.
You can use these `@id`s to access data and metadata throughout the notebook.

In [ ]:
# List all RecordSets in the dataset, with their @id
record_sets = dataset.metadata.record_sets  # This is a list of RecordSet objects
print("Available RecordSets:")
for rs in record_sets:
    print(f"@id: {rs.id} | name: {getattr(rs, 'name', '')}")

# Display all Fields and their @id for each RecordSet
print("\nFields per RecordSet:")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}, name: {getattr(rs, 'name', '')}")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"  Field @id: {field.id} | name: {getattr(field, 'name', '')} | dataType: {getattr(field, 'data_type', '')}")
    else:
        print("  No fields listed.")

## 3. Data Extraction

Extract data from each available RecordSet using only their `@id`.

_Tip: Refer to the above overview for the `@id` you wish to query._

In [ ]:
# Prepare a dictionary of DataFrames from each record set
dataframes = {}

# Collect all record set @id values
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Fields: {df.columns.tolist()}")
        display(df.head())
    else:
        print("  No records found in this RecordSet.")

# For simplicity, let's pick the first available non-empty record set for detailed processing below
main_record_set_id = None
for key, frame in dataframes.items():
    if not frame.empty:
        main_record_set_id = key
        break
if main_record_set_id:
    print(f"\nSelected main RecordSet for further analysis: {main_record_set_id}")
    print("Available columns/fields:", dataframes[main_record_set_id].columns.tolist())
else:
    print("No data available for analysis.")

## 4. Exploratory Data Analysis (EDA)

Let's perform basic cleaning, analysis, and grouping.
We refer to columns by their `@id` (as found in the previous section).

In [ ]:
# ---- Edit the variables below to match IDs/candidates from the previous Overview ----
# We'll attempt to auto-select a numeric field (e.g. age, interval, etc.)
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to find a likely numeric column by type or name (has 'age', 'interval', etc.)
    candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or df[col].dtype in ['int64','float64']]
    if candidates:
        numeric_field_id = candidates[0]
        print(f"Using numeric field candidate: {numeric_field_id}")
    else:
        numeric_field_id = df.select_dtypes(include=[float, int]).columns[0] if not df.select_dtypes(include=[float, int]).empty else df.columns[0]

    # Set threshold for filtering (example: > 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}: (showing top 5)")
    display(filtered_df.head())

    # Z-score normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical field (e.g. 'sex', 'location', 'status'), falling back to the first object or category field
    group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'msi', 'status', 'location', 'histology', 'anatomical', 'metastasis', 'group'])]
    if group_candidates:
        group_field = group_candidates[0]
    else:
        group_field = df.select_dtypes(include=['object', 'category']).columns[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df)
    else:
        print("No suitable group field found for grouping analysis.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize the distribution and group-wise differences in the numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 colorectal cancer dataset using the `mlcroissant` library, referencing all schema entries by their `@id`. We reviewed available record sets and fields, loaded tabular data, applied simple data filtering and normalization, and visualized key patterns. This workflow can be adapted to other Croissant-based datasets—simply substitute the Croissant schema URL and reference the relevant `@id` fields for each step.

*Always verify output and field semantics against the dataset schema and documentation for robust clinical or scientific workflows.*